<a href="https://colab.research.google.com/github/quannca/BTL_XLA/blob/main/improvements/keds/notebook/02_Improved_KEDs_Demo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==========================================
# CELL 1: CÀI ĐẶT THƯ VIỆN CẦN THIẾT
# ==========================================
!pip install -q ftfy regex tqdm gradio timm transformers
!pip install -q git+https://github.com/openai/CLIP.git

import torch
print("--- KIỂM TRA MÔI TRƯỜNG ---")
if torch.cuda.is_available():
    print(f"✅ Tuyệt vời! Đang chạy trên GPU: {torch.cuda.get_device_name(0)}")
else:
    print("ℹ️ Đang chạy trên CPU")

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
--- KIỂM TRA MÔI TRƯỜNG ---
✅ Tuyệt vời! Đang chạy trên GPU: Tesla T4


In [ ]:
# ==========================================
# CELL 2: TẢI CÁC MÔ HÌNH AI (CLIP, PIC2WORD, BLIP)
# ==========================================
import os
import requests
import torch
import torch.nn as nn
from PIL import Image
import clip
from transformers import BlipProcessor, BlipForConditionalGeneration

device = "cuda" if torch.cuda.is_available() else "cpu"

# 1. Định nghĩa mạng ánh xạ MLP 3 lớp của Pic2Word (theo chuẩn bài báo Table B)
class Pic2WordMappingNetwork(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=512, output_dim=768):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Dropout(0.1),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Dropout(0.1),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )
    def forward(self, x):
        return self.network(x)

# 2. Tải mô hình CLIP ViT-L/14 (Mô hình nền tảng)
print("[*] 1/3 Đang tải mô hình CLIP ViT-L/14...")
clip_model, clip_preprocess = clip.load("ViT-L/14", device=device)
clip_model.eval()

# 3. Khởi tạo mạng Pic2Word Mapping Network
print("[*] 2/3 Khởi tạo Mapping Network của Pic2Word...")
mapping_net = Pic2WordMappingNetwork(input_dim=768, hidden_dim=512, output_dim=768).to(device)
mapping_net.eval()

# 4. Tải mô hình BLIP để tạo luồng tri thức KEDs
print("[*] 3/3 Đang tải mô hình BLIP Image Captioning...")
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
blip_model.eval()

print("🎉 ĐÃ TẢI XONG TOÀN BỘ MÔ HÌNH THÀNH CÔNG!")

[*] 1/3 Đang tải mô hình CLIP ViT-L/14...


100%|███████████████████████████████████████| 890M/890M [00:13<00:00, 70.9MiB/s]
/usr/local/lib/python3.13/dist-packages/torch/jit/_serialization.py:176: FutureWarning: `torch.jit.load` is deprecated. Please switch to `torch.export`.
  warnings.warn(


[*] 2/3 Khởi tạo Mapping Network của Pic2Word...
[*] 3/3 Đang tải mô hình BLIP Image Captioning...


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.56k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

... (5 outputs left out)

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  990MB            

model.safetensors: downloading bytes:           |  0.00B            

🎉 ĐÃ TẢI XONG TOÀN BỘ MÔ HÌNH THÀNH CÔNG!


In [ ]:
# ==========================================
# CELL 3: KẾT NỐI GOOGLE DRIVE & NẠP KHO DỮ LIỆU THẬT CỦA BẠN QUÂN
# ==========================================
from google.colab import drive
import os
import json
import torch

# 1. Mount (Kết nối) tới Google Drive của bạn
print("[*] Đang kết nối tới Google Drive...")
drive.mount('/content/drive')

# 2. Đường dẫn tới thư mục COCO_GALLERY của Quân mà bạn vừa thêm lối tắt
drive_folder = '/content/drive/MyDrive/COCO_GALLERY'

features_path = os.path.join(drive_folder, 'gallery_features.pt')
paths_json_path = os.path.join(drive_folder, 'image_paths.json')

# 3. Nạp ma trận đặc trưng và danh sách đường dẫn ảnh
print("[*] Đang nạp dữ liệu thật từ Drive...")
gallery_features = torch.load(features_path, map_location=device)
gallery_features = gallery_features / gallery_features.norm(dim=-1, keepdim=True)

with open(paths_json_path, 'r', encoding='utf-8') as f:
    gallery_paths = json.load(f)

print(f"🎉 NẠP THÀNH CÔNG KHO DỮ LIỆU THẬT CỦA QUÂN!")
print(f"✅ Số lượng ảnh trong kho: {len(gallery_paths)} ảnh")
print(f"✅ Kích thước ma trận đặc trưng: {gallery_features.shape}")

[*] Đang kết nối tới Google Drive...
Mounted at /content/drive
[*] Đang nạp dữ liệu thật từ Drive...
🎉 NẠP THÀNH CÔNG KHO DỮ LIỆU THẬT CỦA QUÂN!
✅ Số lượng ảnh trong kho: 5000 ảnh
✅ Kích thước ma trận đặc trưng: torch.Size([5000, 768])


In [ ]:
# ==========================================
# CELL 4: BỘ MÁY TÌM KIẾM DUAL-STREAM (ĐÃ SỬA LỖI HALF/FLOAT CHUẨN GPU)
# TỰ ĐỘNG TẢI ẢNH COCO VÀO RAM HIỂN THỊ NÉT CĂNG
# ==========================================
import requests
from io import BytesIO

def get_blip_caption(image: Image.Image) -> str:
    """Luồng Tri thức KEDs: Tự động trích xuất ngữ cảnh chi tiết của ảnh gốc qua BLIP"""
    inputs = blip_processor(image, return_tensors="pt").to(device)
    with torch.no_grad():
        out = blip_model.generate(**inputs, max_new_tokens=30)
    caption = blip_processor.decode(out[0], skip_special_tokens=True)
    return caption

def load_image_from_coco(raw_path: str):
    """Tải trực tiếp ảnh từ máy chủ COCO về RAM để Gradio hiển thị mượt mà"""
    raw_path = raw_path.replace("\\", "/")
    if "val2017/" in raw_path:
        filename = raw_path.split("val2017/")[-1]
        url = f"http://images.cocodataset.org/val2017/{filename}"
        try:
            resp = requests.get(url, timeout=5)
            if resp.status_code == 200:
                return Image.open(BytesIO(resp.content)).convert("RGB")
        except Exception:
            pass
    return Image.new("RGB", (300, 300), color="gray")

def search_dual_stream(ref_image: Image.Image, text_query: str, alpha: float = 0.5, top_k: int = 3):
    img_tensor = clip_preprocess(ref_image).unsqueeze(0).to(device)

    # -------------------------------------------------------------
    # LUỒNG 1: PIC2WORD BASELINE (Đồng bộ kiểu dữ liệu với CLIP)
    # -------------------------------------------------------------
    with torch.no_grad():
        img_feature = clip_model.encode_image(img_tensor).float() # Chuyển sang Float
        pseudo_token = mapping_net(img_feature).type(clip_model.dtype) # Ép chuẩn kiểu CLIP

        prompt_text = f"a photo of something, {text_query}"
        tokenized = clip.tokenize([prompt_text]).to(device)

        text_emb = clip_model.token_embedding(tokenized).type(clip_model.dtype)
        text_emb[:, 4, :] = pseudo_token

        x = text_emb + clip_model.positional_embedding.type(clip_model.dtype)
        x = x.permute(1, 0, 2)
        x = clip_model.transformer(x)
        x = x.permute(1, 0, 2)
        x = clip_model.ln_final(x).type(clip_model.dtype)
        pic2word_feat = x[torch.arange(x.shape[0]), tokenized.argmax(dim=-1)] @ clip_model.text_projection
        pic2word_feat = pic2word_feat / pic2word_feat.norm(dim=-1, keepdim=True)

        # Đảm bảo gallery_features cũng đúng kiểu dữ liệu
        g_feats = gallery_features.to(device).type(pic2word_feat.dtype)
        sim_pic2word = (pic2word_feat @ g_feats.T).squeeze(0)

    # -------------------------------------------------------------
    # LUỒNG 2: KEDS KNOWLEDGE-ENHANCED STREAM (Dùng Caption BLIP)
    # -------------------------------------------------------------
    caption = get_blip_caption(ref_image)
    enhanced_prompt = f"a photo of {caption}, but {text_query}"
    with torch.no_grad():
        tokenized_keds = clip.tokenize([enhanced_prompt], truncate=True).to(device)
        keds_feat = clip_model.encode_text(tokenized_keds)
        keds_feat = keds_feat / keds_feat.norm(dim=-1, keepdim=True)

        sim_knowledge = (keds_feat @ g_feats.T).squeeze(0)

    # -------------------------------------------------------------
    # KẾT HỢP DUAL-STREAM THEO TRỌNG SỐ ALPHA
    # -------------------------------------------------------------
    sim_keds_final = alpha * sim_pic2word + (1.0 - alpha) * sim_knowledge

    def get_top_pil_images(sim_vector):
        top_indices = torch.topk(sim_vector, k=min(top_k, len(gallery_paths))).indices.cpu().numpy()
        return [load_image_from_coco(gallery_paths[idx]) for idx in top_indices]

    results_pic2word = get_top_pil_images(sim_pic2word)
    results_keds = get_top_pil_images(sim_keds_final)

    return caption, results_pic2word, results_keds

print("✅ ĐÃ FIX TRIỆT ĐỂ LỖI DTYPE HALF/FLOAT!")

✅ ĐÃ FIX TRIỆT ĐỂ LỖI DTYPE HALF/FLOAT!


In [1]:
# ==========================================
# CELL 5: GIAO DIỆN WEB GRADIO SO SÁNH TRỰC DIỆN
# ==========================================
import gradio as gr

def run_demo(image, text_query, alpha):
    if image is None:
        return "⚠️ Vui lòng tải một bức ảnh lên!", [], []
    if not text_query or not text_query.strip():
        return "⚠️ Vui lòng nhập câu lệnh mô tả thay đổi!", [], []

    caption, p2w_imgs, keds_imgs = search_dual_stream(
        ref_image=image,
        text_query=text_query,
        alpha=alpha,
        top_k=3
    )

    info_markdown = f"""
    ### 💡 Tri thức BLIP trích xuất từ ảnh gốc:
    > **"{caption}"**

    *(Hệ thống KEDs đã dùng ngữ cảnh chi tiết này kết hợp với câu lệnh của bạn để tìm kiếm chính xác hơn!)*
    """
    return info_markdown, p2w_imgs, keds_imgs

with gr.Blocks(title="BTL Xử Lý Ảnh - PTIT") as demo:
    gr.Markdown("# 🚀 BTL XỬ LÝ ẢNH - PTIT: HỆ THỐNG TRUY XUẤT ẢNH ZS-CIR")
    gr.Markdown("### So sánh trực tiếp: **Pic2Word Baseline** vs **KEDs Cải tiến (Dual-stream)**")

    with gr.Row():
        with gr.Column(scale=1):
            input_img = gr.Image(type="pil", label="1. Tải ảnh tham chiếu (Reference Image)")
            query_txt = gr.Textbox(placeholder="Ví dụ: change to cat, origami style...", label="2. Yêu cầu biến đổi (Text Modifier)")
            alpha_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.5, step=0.1,
                                     label="Trọng số Alpha (0.5 = Cân bằng giữa Pic2Word và Luồng Tri thức)")
            btn_search = gr.Button("🔍 Bắt đầu tìm kiếm & So sánh", variant="primary")

            caption_output = gr.Markdown()

        with gr.Column(scale=2):
            gr.Markdown("### 📊 KẾT QUẢ SO SÁNH ĐỐI CHỨNG")
            with gr.Row():
                with gr.Column():
                    gr.Markdown("#### ❌ Baseline: Pic2Word (Chỉ dùng 1 Token)")
                    gallery_p2w = gr.Gallery(label="Top kết quả Pic2Word", columns=1, height="auto")
                with gr.Column():
                    gr.Markdown("#### ✅ Cải tiến: KEDs (Dual-stream)")
                    gallery_keds = gr.Gallery(label="Top kết quả KEDs Cải tiến", columns=1, height="auto")

    btn_search.click(
        fn=run_demo,
        inputs=[input_img, query_txt, alpha_slider],
        outputs=[caption_output, gallery_p2w, gallery_keds]
    )

# Khởi chạy giao diện web
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://c3a5413050bac60723.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://c3a5413050bac60723.gradio.live


In [ ]:
# KIỂM TRA ĐƯỜNG DẪN ẢNH CỦA BẠN QUÂN
print("3 đường dẫn ảnh đầu tiên trong file của Quân:")
for p in gallery_paths[:3]:
    print(p)

3 đường dẫn ảnh đầu tiên trong file của Quân:
val2017/000000000139.jpg
val2017/000000000285.jpg
val2017/000000000632.jpg
